# 05 · Tools：先跑通一次可观察的工具调用

这一节暂时不做意图分流，只验证最小工具闭环：

HumanMessage → Agent → AIMessage(tool call) → ToolNode → ToolMessage → Agent → final

练习使用 TEU 换算作为确定性工具。只有最终数字不算通关，必须看到工具名、参数、调用 ID 和 ToolMessage。

本节末尾补充 MCP 配置示例：工具也可以来自独立进程，加载后仍走同一套 Agent ↔ ToolNode 闭环。


In [ ]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Annotated, TypedDict, Literal

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / ".env.example").exists():
        ROOT = candidate
        break
os.chdir(ROOT)
load_dotenv(ROOT / ".env")

missing_llm = [
    name for name in ("LLM_BASE_URL", "LLM_MODEL")
    if not (os.getenv(name) or "").strip()
]
if missing_llm:
    raise ValueError(
        "真实 LLM 教学路径必须配置 .env，缺少：" + ", ".join(missing_llm)
    )
print("cwd =", ROOT)
print("mode = live LLM required")

## 1. 先把工具当普通函数验收


In [2]:
TEU_PER_EQUIPMENT = {"20GP": 1, "40GP": 2, "40HQ": 2}


@tool
def calculate_teu(equipment: str, quantity: int) -> int:
    """按课堂约定换算 TEU。支持 20GP、40GP、40HQ。"""
    key = equipment.strip().upper()
    if key not in TEU_PER_EQUIPMENT:
        raise ValueError(f"不支持的箱型: {equipment}")
    if quantity < 0:
        raise ValueError("quantity 不能为负数")
    return TEU_PER_EQUIPMENT[key] * quantity


assert calculate_teu.invoke({"equipment": "40HQ", "quantity": 3}) == 6
assert calculate_teu.invoke({"equipment": "20GP", "quantity": 2}) == 2
print("05 tool unit ok: 3×40HQ=6, 2×20GP=2")


05 tool unit ok: 3×40HQ=6, 2×20GP=2


## 2. 把工具接进 LangGraph

Agent 节点只负责提出工具调用或生成最终回答；ToolNode 才负责校验参数、执行函数并把 ToolMessage 追加到 messages。


In [ ]:
class ToolState(TypedDict):
    messages: Annotated[list, add_messages]


def make_llm():
    from langchain_openai import ChatOpenAI

    return ChatOpenAI(
        model=os.environ["LLM_MODEL"],
        api_key=os.getenv("LLM_API_KEY") or "not-required",
        base_url=os.environ["LLM_BASE_URL"],
        temperature=0,
    )


llm_with_tools = make_llm().bind_tools([calculate_teu])


def agent(state: ToolState) -> dict:
    response = llm_with_tools.invoke([
        SystemMessage(content="所有 TEU 计算必须调用 calculate_teu；收到工具结果后再汇总。"),
        *state["messages"],
    ])
    return {"messages": [response]}


def after_agent(state: ToolState) -> Literal["tools", "end"]:
    return "tools" if getattr(state["messages"][-1], "tool_calls", None) else "end"


builder = StateGraph(ToolState)
builder.add_node("agent", agent)
builder.add_node("tools", ToolNode([calculate_teu]))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", after_agent, {"tools": "tools", "end": END})
builder.add_edge("tools", "agent")
graph = builder.compile()

try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("PNG 不可用，打印 Mermaid：", exc)
    print(graph.get_graph().draw_mermaid())

## 3. 运行真实工具闭环


In [ ]:
result = graph.invoke({
    "messages": [HumanMessage(content="请用工具计算 3×40HQ + 2×20GP 的 TEU，并汇总。")]
})
tool_messages = [message for message in result["messages"] if isinstance(message, ToolMessage)]
print("message_types =", [type(message).__name__ for message in result["messages"]])
print("tool_messages =", [(message.name, message.tool_call_id, message.content) for message in tool_messages])
assert len(tool_messages) == 2
assert all(message.name == "calculate_teu" for message in tool_messages)
assert sum(int(str(message.content)) for message in tool_messages) == 8
print("05 tools live path ok")

## 4*. 把 MCP 放进 LangGraph

MCP **不是**图里的新节点。拓扑仍是第 2 节那张图：

`START → agent ⇄ tools → END`

变的只是 `tools` 从哪来：本地 `@tool` 换成 `MultiServerMCPClient.get_tools()`。MCP 进入图的两个位置：

1. `bind_tools(mcp_tools)`：agent 节点才能提出 MCP tool call
2. `ToolNode(mcp_tools)`：tools 节点才真正去调 MCP Server

两种常见 transport：`stdio` 拉起本地进程；`http` 连接已启动的远程服务。


In [ ]:
# MCP 客户端配置：声明怎么连 Server，而不是在 notebook 里再写一遍 @tool。
# 接入时需要：uv add langchain-mcp-adapters
MCP_SERVERS = {
    "teu": {
        "transport": "stdio",
        "command": "python",
        "args": [str(ROOT / "servers" / "teu_mcp.py")],
    },
    "schedule": {
        "transport": "http",
        "url": "http://localhost:8000/mcp",
        "headers": {"Authorization": "Bearer YOUR_TOKEN"},
    },
}

assert MCP_SERVERS["teu"]["transport"] == "stdio"
assert MCP_SERVERS["teu"]["command"] == "python"
assert Path(MCP_SERVERS["teu"]["args"][0]).name == "teu_mcp.py"
assert MCP_SERVERS["schedule"]["transport"] == "http"
assert MCP_SERVERS["schedule"]["url"].endswith("/mcp")
print("mcp servers =", list(MCP_SERVERS))
print("teu transport =", MCP_SERVERS["teu"]["transport"], MCP_SERVERS["teu"]["args"])
print("schedule url =", MCP_SERVERS["schedule"]["url"])
print("05 mcp config ok")


对照第 2 节，只有工具来源不同（需 `uv add langchain-mcp-adapters`，且 stdio 脚本或 http 服务已启动）：

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(MCP_SERVERS)
mcp_tools = await client.get_tools()   # 得到与 @tool 同类的 LangChain tools

llm_with_mcp = make_llm().bind_tools(mcp_tools)

def mcp_agent(state: ToolState) -> dict:
    response = llm_with_mcp.invoke([
        SystemMessage(content="优先调用 MCP 工具完成计算，收到结果后再汇总。"),
        *state["messages"],
    ])
    return {"messages": [response]}

mcp_builder = StateGraph(ToolState)
mcp_builder.add_node("agent", mcp_agent)
mcp_builder.add_node("tools", ToolNode(mcp_tools))  # MCP 在这里进入图
mcp_builder.add_edge(START, "agent")
mcp_builder.add_conditional_edges("agent", after_agent, {"tools": "tools", "end": END})
mcp_builder.add_edge("tools", "agent")
mcp_graph = mcp_builder.compile()

不要在已经 `compile()` 过的 `builder` 上再 `add_node`。新建一张图，把 MCP tools 传进去。

In [ ]:
def make_tool_graph(tools):
    """同一张图：换 tools 列表就能从本地 @tool 切到 MCP。"""
    llm = make_llm().bind_tools(tools)

    def agent_node(state: ToolState) -> dict:
        response = llm.invoke([
            SystemMessage(content="所有计算必须调用工具；收到工具结果后再汇总。"),
            *state["messages"],
        ])
        return {"messages": [response]}

    g = StateGraph(ToolState)
    g.add_node("agent", agent_node)
    g.add_node("tools", ToolNode(tools))
    g.add_edge(START, "agent")
    g.add_conditional_edges("agent", after_agent, {"tools": "tools", "end": END})
    g.add_edge("tools", "agent")
    return g.compile()


# 第 2 节写法：make_tool_graph([calculate_teu])
# MCP 写法：  make_tool_graph(await MultiServerMCPClient(MCP_SERVERS).get_tools())
same_graph = make_tool_graph([calculate_teu])
print("graph nodes =", list(same_graph.get_graph().nodes))
print("MCP 接入点 = bind_tools(tools) + ToolNode(tools)")
print("05 mcp graph wiring ok")


对应的 stdio Server 可用 FastMCP 暴露工具，例如 `servers/teu_mcp.py`（`MCP_SERVERS["teu"]["args"]` 指向这个文件）：

In [ ]:

from fastmcp import FastMCP

TEU_PER_EQUIPMENT = {"20GP": 1, "40GP": 2, "40HQ": 2}
mcp = FastMCP("TEU")


@mcp.tool()
def calculate_teu(equipment: str, quantity: int) -> int:
    """按课堂约定换算 TEU。支持 20GP、40GP、40HQ。"""
    key = equipment.strip().upper()
    if key not in TEU_PER_EQUIPMENT:
        raise ValueError(f"不支持的箱型: {equipment}")
    if quantity < 0:
        raise ValueError("quantity 不能为负数")
    return TEU_PER_EQUIPMENT[key] * quantity


if __name__ == "__main__":
    mcp.run(transport="stdio")

要点：MCP 替换的是 **tools 列表**，不是 Graph 拓扑。预置写法 `create_agent(model, mcp_tools)` 内部也是同一张 agent ⇄ tools 图。


## 20 分钟双人练习

使用自己的 AI Coding 工具完成：

1. 新增一个只读工具，输入和返回值必须有明确 schema。
2. 把它接入 StateGraph 的 Agent ↔ ToolNode 循环。
3. 保存一条 AIMessage(tool call) 与对应 ToolMessage。
4. 准备一个非法参数失败样例；失败不得伪装成成功回答。

交付：图结构、一次成功 Trace、一次失败证据。20 分钟后随机抽一组展示。


<!-- codex:checklist -->
---

## 练习任务 Checklist

完成后逐项勾选：

- [ ] 把 `calculate_teu` 当普通函数完成成功与非法参数测试。
- [ ] 找到 AIMessage 中的 tool call 名称、参数和调用 ID。
- [ ] 找到对应 ToolMessage，并核对 `tool_call_id`。
- [ ] 证明 Agent → ToolNode → Agent 闭环得到 8 TEU。
- [ ] 读懂 MCP 配置里 `stdio` 与 `http` 两种连接方式，以及加载后如何接入 ToolNode。
- [ ] 新增一个只读工具，并准备成功与失败各一条 Trace。

**交付证据：**工具单测、AIMessage/ToolMessage 对照、stream event 摘要。